# 70 · Governance — DataHub catalog, lineage & vocabulary

**DataHub is the mesh's metadata catalog.** Every other notebook in this library *queries
data*; this one queries the **metadata about that data** — what datasets exist, across which
engines, who owns them, what the columns mean, and — the governance payoff — **where each
dataset's data came from and what depends on it**.

Think of it as the map, not the territory. The lakehouse (`10`/`11`), Trino (`20`), the Tier-2
stores (`22`) and the vector stores (`30`–`32`) each hold *data*; DataHub holds one searchable
description of *all of them at once*, kept current by the mesh's emitters (`datahub_emit.py`
runs from Dagster). When the operator or an agent needs to answer "is there a dataset about X,
and can I trust it?", this is the surface they hit — and they hit it exactly the way this
notebook does: programmatically, and **read-only**.

### DataHub's model: entities and aspects

DataHub describes the platform with two ideas.

**Entities** are the things worth cataloging, each with a stable URN:

| entity | what it is | example |
|--------|-----------|---------|
| **dataset** | a table / collection / index on some platform | `(iceberg, dbt.mart_spotify_audio, PROD)` |
| **domain** | a business grouping of datasets | `Music`, `Health`, `Docs & RAG` |
| **glossary term** | a defined word in the shared vocabulary | `Danceability`, `Chunk`, `Agent` |
| **application / data product** | a shipped capability over datasets | a platform service |
| **dashboard / chart / dataJob** | BI + pipeline nodes that consume datasets | Lightdash charts, dbt build jobs |

**Aspects** are the facets hung on an entity — its *schema* (columns + types), *ownership*,
*tags*, *glossary terms*, *domain*, and its **lineage** (the upstream/downstream edges). One
dataset URN, many aspects.

> **Read-only, throughout.** Everything here is a search or a fetch — `me`, `dataset(...)`,
> `searchAcrossEntities`, `searchAcrossLineage`, `listDomains`. Nothing writes metadata: no tag
> applied, no term added, no ownership changed. As in notebooks `20`/`22`, because we create
> nothing there is **no cleanup section**. We also never *assume* a URN — we **search for real
> datasets first**, then inspect the ones we actually found.

## Setup

The DataHub client — `acryl-datahub` — is **not** in the singleuser base image (which ships
`polars`, `s3fs`, `pyarrow`, `duckdb`, `fastavro`), so we install it here. `polars` — used to
render every result frame, exactly as in notebooks `20`/`22` — already ships in the image.

We use the SDK's `DataHubGraph` client throughout: it wraps the GMS REST + GraphQL API and is
the same client `datahub_emit.py` and the operator use. Every query below goes through its
`execute_graphql` method — the cleanest way to pull the exact aspects we want in one round trip.

In [1]:
%pip install -q acryl-datahub


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect

Connection is **env-driven**, the same pattern the query notebooks use. The committed default is
the **in-cluster** GMS service URL
(`http://datahub-datahub-gms.data-mesh.svc.cluster.local:8080`); a run overrides `DATAHUB_GMS_URL`
via env without editing the notebook. Auth is a **bearer token** read from `DATAHUB_TOKEN` (never
committed, never echoed) — in the cluster it comes from the `weyland` namespace secret
`datahub-token`.

`gql(query, variables)` is our one helper: run a GraphQL query against GMS and hand back the
parsed JSON. We prove the connection two ways — **who** the token authenticates as, and **how
many** entities the catalog currently holds.

In [2]:
import os
import polars as pl
from datahub.ingestion.graph.client import DataHubGraph, DataHubGraphConfig

# committed default = in-cluster GMS DNS; a run overrides DATAHUB_GMS_URL via env
GMS_URL = os.environ.get("DATAHUB_GMS_URL", "http://datahub-datahub-gms.data-mesh.svc.cluster.local:8080")
TOKEN   = os.environ["DATAHUB_TOKEN"]   # bearer token, never committed / never echoed

graph = DataHubGraph(DataHubGraphConfig(server=GMS_URL, token=TOKEN))

def gql(query, variables=None):
    """Run a GraphQL query against GMS, return the parsed `data` payload."""
    return graph.execute_graphql(query, variables=variables or {})

# prove the connection: who does this token authenticate as, and how much is cataloged?
me = gql("query { me { corpUser { urn username } } }")["me"]["corpUser"]
total = gql('query { searchAcrossEntities(input: {query: "*", start: 0, count: 0}) { total } }')["searchAcrossEntities"]["total"]

print("GMS               :", GMS_URL.split("//", 1)[-1])
print("authenticated as  :", me["username"])
print("entities cataloged:", total)

GMS               : datahub-datahub-gms.data-mesh.svc.cluster.local:8080
authenticated as  : emangini
entities cataloged: 9985


## Search — what datasets exist?

The first thing a catalog buys you is **discovery**: ask for a keyword and get back matching
datasets across *every* platform at once, without knowing where they live. `searchAcrossEntities`
is that entry point. Here we search `mart` — the dbt gold-layer tables — and render each hit's
platform, name, and URN. (Swap the keyword and the same call answers "is there anything about
spotify / health / chunks?".)

Look at the platforms in the result: the mesh catalogs the *same* logical table as it appears on
several engines — the Iceberg lakehouse copy, the dbt model, a DuckDB extract — each a distinct
URN. That is the catalog showing you the whole footprint of one dataset.

In [3]:
res = gql("""
query($q: String!) {
  searchAcrossEntities(input: {types: [DATASET], query: $q, start: 0, count: 15}) {
    total
    searchResults { entity { urn ... on Dataset { name platform { name } } } }
  }
}""", {"q": "mart"})

hits = res["searchAcrossEntities"]
rows = [{"platform": s["entity"]["platform"]["name"],
         "name":     s["entity"]["name"],
         "urn":      s["entity"]["urn"]}
        for s in hits["searchResults"]]
print(f'{hits["total"]} datasets match "mart"; first {len(rows)} shown')
pl.DataFrame(rows)

35 datasets match "mart"; first 15 shown


platform,name,urn
str,str,str
"""iceberg""","""mart_artist_popularity__dbt_tm…","""urn:li:dataset:(urn:li:dataPla…"
"""iceberg""","""mart_artist_popularity""","""urn:li:dataset:(urn:li:dataPla…"
"""iceberg""","""mart_genre_audio_profile""","""urn:li:dataset:(urn:li:dataPla…"
"""iceberg""","""mart_country_health""","""urn:li:dataset:(urn:li:dataPla…"
"""iceberg""","""mart_fma_genre_tree""","""urn:li:dataset:(urn:li:dataPla…"
…,…,…
"""dbt""","""mart_genre_audio_profile""","""urn:li:dataset:(urn:li:dataPla…"
"""dbt""","""mart_spotify_audio""","""urn:li:dataset:(urn:li:dataPla…"
"""dbt""","""mart_artist_popularity""","""urn:li:dataset:(urn:li:dataPla…"


## Inspect a dataset — schema, ownership, tags, domain

Pick one real URN from the search — `iceberg.dbt.mart_spotify_audio`, a dbt gold mart — and pull
its **aspects** in a single query: the **schema** (columns + native types), who **owns** it, what
**tags** and **glossary terms** are attached, and which **domain** it belongs to. This is the
"what is this, and can I trust it?" view an analyst hits before querying the table for real.

The schema `fieldPath` values carry DataHub's nested type-encoding
(`[version=2.0].[type=struct]…`); we show the leaf column name and its native type — the columns
you would actually `SELECT`.

In [4]:
MART = "urn:li:dataset:(urn:li:dataPlatform:iceberg,dbt.mart_spotify_audio,PROD)"

res = gql("""
query($urn: String!) {
  dataset(urn: $urn) {
    name platform { name }
    domain { domain { properties { name } } }
    ownership { owners { owner { ... on CorpUser { urn } ... on CorpGroup { urn } }
                        ownershipType { info { name } } } }
    tags { tags { tag { name } } }
    glossaryTerms { terms { term { properties { name } } } }
    schemaMetadata { fields { fieldPath nativeDataType } }
  }
}""", {"urn": MART})

ds = res["dataset"]
domain = ds["domain"]["domain"]["properties"]["name"] if ds.get("domain") else None
owners = [f'{o["owner"]["urn"].split(":")[-1]} ({o["ownershipType"]["info"]["name"]})'
          for o in (ds.get("ownership") or {}).get("owners", [])]
tags  = [t["tag"]["name"] for t in (ds.get("tags") or {}).get("tags", [])]
terms = [t["term"]["properties"]["name"] for t in (ds.get("glossaryTerms") or {}).get("terms", [])]

print("dataset :", ds["name"], "on", ds["platform"]["name"])
print("domain  :", domain)
print("owners  :", owners)
print("tags    :", tags)
print("terms   :", terms or "(none applied at the dataset level)")

fields = (ds.get("schemaMetadata") or {}).get("fields", [])
pl.DataFrame([{"column": f["fieldPath"].split(".")[-1], "type": f["nativeDataType"]} for f in fields])

dataset : mart_spotify_audio on iceberg
domain  : Music
owners  : ['weyland (Technical Owner)']
tags    : ['lakehouse', 'spotify', 'mart']
terms   : (none applied at the dataset level)


column,type
str,str
"""track_id""","""string"""
"""track_genre""","""string"""
"""danceability""","""double"""
"""energy""","""double"""
"""key""","""double"""
…,…
"""acousticness""","""double"""
"""instrumentalness""","""double"""
"""liveness""","""double"""


## Lineage — where did this come from, and what depends on it?

This is the governance payoff. **Lineage** is the graph of upstream/downstream edges between
datasets (and the pipeline jobs, charts and dashboards around them), and it answers the two
questions a data platform lives or dies by:

- **Provenance — "where did this data come from?"** Walk *upstream* from a dataset to the sources
  and transforms that produced it. Below: the `weyland_chunks` vector collection in Qdrant, traced
  back through the Dagster assets (`source_document` → `chunks` → `embeddings` → `qdrant_write`)
  that built its embeddings. If you are about to trust a retrieval result, this is how you see what
  fed it.
- **Impact — "what breaks if I change this?"** Walk *downstream* to everything that consumes a
  dataset. In the second cell, `mart_spotify_audio` fans out to the Feast feature source, the
  downstream `mart_genre_audio_profile`, its Trino view, and the Lightdash charts/dashboards built
  on it — the blast radius of any schema change.

`searchAcrossLineage` returns each edge with its **degree** (`hops` — 1 = direct neighbour, 2+ =
indirect). Reading these two frames is impact analysis you would otherwise reconstruct by hand
across five systems.

In [5]:
LINEAGE = """
query($urn: String!, $dir: LineageDirection!) {
  searchAcrossLineage(input: {urn: $urn, direction: $dir, query: "*", start: 0, count: 25}) {
    total
    searchResults { degree entity { urn type ... on Dataset { name platform { name } } } }
  }
}"""

def _label(entity):
    """Readable name: 'platform:name' for datasets; the trailing id for charts/jobs/dashboards."""
    if entity.get("name"):
        plat = (entity.get("platform") or {}).get("name", "")
        return f'{plat}:{entity["name"]}' if plat else entity["name"]
    return entity["urn"].split(",")[-1].rstrip(")")

def lineage_frame(urn, direction):
    res = gql(LINEAGE, {"urn": urn, "dir": direction})["searchAcrossLineage"]
    rows = [{"hops": s["degree"], "type": s["entity"]["type"], "entity": _label(s["entity"])}
            for s in res["searchResults"]]
    return res["total"], pl.DataFrame(rows).sort("hops") if rows else pl.DataFrame(rows)

# PROVENANCE: where do the weyland_chunks vectors come from?
up_total, up = lineage_frame("urn:li:dataset:(urn:li:dataPlatform:qdrant,weyland_chunks,PROD)", "UPSTREAM")
print(f"weyland_chunks (qdrant) -> {up_total} upstream entities: where its vectors came from")
up

weyland_chunks (qdrant) -> 6 upstream entities: where its vectors came from


hops,type,entity
i64,str,str
1,"""DATASET""","""dagster:qdrant_write"""
2,"""DATASET""","""dagster:embeddings"""
2,"""DATASET""","""dagster:hash_check"""
2,"""DATASET""","""dagster:source_document"""
3,"""DATASET""","""dagster:chunks"""
3,"""DATASET""","""dagster:content_hash"""


In [6]:
# IMPACT: what depends on the spotify-audio mart?
down_total, down = lineage_frame(MART, "DOWNSTREAM")
print(f"mart_spotify_audio -> {down_total} downstream entities: what a schema change would touch")
down

mart_spotify_audio -> 13 downstream entities: what a schema change would touch


hops,type,entity
i64,str,str
1,"""DATASET""","""trino:mart_spotify_audio"""
2,"""DATASET""","""dbt:mart_genre_audio_profile"""
2,"""DATASET""","""postgres:track_audio_features"""
2,"""CHART""","""3494af0c-0b3e-4a75-92a3-ea970b…"
2,"""CHART""","""abefe843-2107-47ae-a9ac-5c0984…"
…,…,…
2,"""DATA_JOB""","""iceberg.dbt.weyland.mart_spoti…"
3,"""DATASET""","""trino:mart_genre_audio_profile"""
3,"""DASHBOARD""","""46a87878-9962-4f6d-86f6-b7393c…"


## Domains & glossary — the governed vocabulary

Search and lineage are about *finding* and *tracing*; **domains** and the **glossary** are about
*shared meaning*. A **domain** groups datasets by business area (so "everything Music" or
"everything Health" is one click, not a keyword guess); a **glossary term** is a word the platform
has agreed on and defined once (so `Danceability` or `Chunk` means the same thing to every team,
dashboard, and agent).

The domain frame also carries each domain's **entity count** — a rough map of where the mesh's
mass actually sits.

In [7]:
res = gql("""
query {
  listDomains(input: {start: 0, count: 30}) {
    total
    domains { properties { name description } entities(input: {start: 0, count: 0}) { total } }
  }
}""")["listDomains"]

print(f'{res["total"]} governed domains:')
pl.DataFrame([{"domain":      d["properties"]["name"],
               "entities":    d["entities"]["total"],
               "description": d["properties"].get("description")}
              for d in res["domains"]]).sort("entities", descending=True)

6 governed domains:


domain,entities,description
str,i64,str
"""Platform & Ops""",4113,"""Operational, observability, an…"
"""Music""",823,"""Music domain — datasets, dbt m…"
"""Health""",402,"""Health / wellness / personalit…"
"""ML & Modeling""",73,"""Model registry and trained mod…"
"""Docs & RAG""",12,"""The platform documentation cor…"
"""AIDLC Knowledge""",6,"""The AIDLC knowledge base — eng…"


In [8]:
res = gql("""
query($q: String!) {
  searchAcrossEntities(input: {types: [GLOSSARY_TERM], query: $q, start: 0, count: 12}) {
    total
    searchResults { entity { ... on GlossaryTerm { properties { name description } } } }
  }
}""", {"q": "*"})["searchAcrossEntities"]

print(f'{res["total"]} glossary terms define the mesh vocabulary; a sample:')
pl.DataFrame([{"term":       s["entity"]["properties"]["name"],
               "definition": s["entity"]["properties"].get("description")}
              for s in res["searchResults"]])

612 glossary terms define the mesh vocabulary; a sample:


term,definition
str,str
"""Acousticness""","""Confidence (0–1) the track is …"
"""Agreeableness""","""OCEAN trait: compassion, coope…"
"""Agent""","""Autonomous / agentic task exec…"
"""Authentication / SSO""","""Identity provider and single s…"
"""Authorization""","""Data-plane access control and …"
…,…
"""Chat Interface""","""Conversational front end."""
"""CI / CD""","""Build and deployment pipelines…"
"""Code Quality""","""Static analysis and quality ga…"


## When to reach for the catalog

**Reach for DataHub when the question is *about* the data, not *in* it:**

- **Discovery** — "is there a dataset about X, and where does it live?" One search spans every
  platform; you don't need to know which store holds it. Cheaper than opening five notebooks.
- **Trust & governance** — before using a table, check its owner, domain, tags, and the glossary
  definitions of its columns. The catalog is where "what does this field *mean*?" is answered.
- **Impact analysis** — "what breaks if I change this schema / drop this column / retire this mart?"
  Walk downstream lineage and read the blast radius instead of guessing.
- **Provenance** — "where did this number come from?" Walk upstream to the sources and transforms.
  Essential when a retrieval or a report looks wrong and you need to trust its inputs.

**Reach for the store directly (the other notebooks) when you want the *data itself*** — to run a
`SELECT`, scan a table, read a vector, join across engines. DataHub will tell you a table exists,
who owns it, and what feeds it; it will **not** return its rows. It is the index, not the pages.

| you want to… | use |
|--------------|-----|
| find a dataset · trace lineage · check ownership & meaning | **DataHub** (this notebook) |
| join across the catalog-connected engines, ad hoc | **Trino** (`20`) |
| query a specialist store on its own protocol | **native client** (`22`) |
| read vectors / do similarity search | **Qdrant / Weaviate / LanceDB** (`30`–`32`) |

The catalog is the mesh's answer to "what do we have, can I trust it, and what happens if I touch
it?" — the map you read *before* you query the territory.